# Von Mises kappa slider

Move the `kappa` slider to draw 1000 fresh samples from `numpyro.distributions.VonMises(0.0, kappa)` and visualize the density around the unit circle. The red direction marks the mean angle at zero.

In [1]:
import numpy as np
import jax
import jax.numpy as jnp
import numpyro.distributions as dist
import matplotlib.pyplot as plt
from IPython.display import display
from ipywidgets import FloatSlider, VBox, interactive_output
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize

plt.rcParams["figure.figsize"] = (7, 7)
plt.rcParams["axes.facecolor"] = "white"

In [5]:
rng_key = jax.random.PRNGKey(0)


def vonmises_pdf(theta, kappa, loc=0.0):
    vm = dist.VonMises(float(loc), float(kappa))
    theta_array = jnp.asarray(theta)
    return np.asarray(jnp.exp(vm.log_prob(theta_array)))


KAPPA_MIN = 0.01
KAPPA_MAX = 1000.0
PDF_DENSITY_MIN = 0.0
PDF_DENSITY_MAX = float(vonmises_pdf(np.array([0.0]), KAPPA_MAX)[0])
PDF_DENSITY_NORM = Normalize(vmin=PDF_DENSITY_MIN, vmax=PDF_DENSITY_MAX)


def draw_vonmises(kappa):
    global rng_key

    loc = 0.0
    n_samples = 1000
    kappa = float(kappa)
    rng_key, sample_key = jax.random.split(rng_key)

    vm = dist.VonMises(loc, kappa)
    samples = np.asarray(vm.sample(sample_key, sample_shape=(n_samples,)))

    theta = np.linspace(-np.pi, np.pi, 721)
    pdf = vonmises_pdf(theta, kappa, loc)

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.set_aspect("equal", adjustable="box")

    # Color the circumference by the Von Mises pdf.
    circle_xy = np.column_stack([np.cos(theta), np.sin(theta)])
    segments = np.stack([circle_xy[:-1], circle_xy[1:]], axis=1)
    line = LineCollection(segments, cmap="viridis", norm=PDF_DENSITY_NORM)
    line.set_array(pdf[:-1])
    line.set_linewidth(9)
    line.set_capstyle("round")
    ax.add_collection(line)

    # Draw sampled angles near the circumference with slight radial jitter.
    jitter_rng = np.random.default_rng()
    sample_radius = 0.72 + 0.20 * jitter_rng.random(n_samples)
    ax.scatter(
        sample_radius * np.cos(samples),
        sample_radius * np.sin(samples),
        s=7,
        color="#2563eb",
        alpha=0.16,
        linewidths=0,
    )

    # Show the pdf shape on a fixed density scale across slider values.
    pdf_scaled = np.clip(
        (pdf - PDF_DENSITY_MIN) / (PDF_DENSITY_MAX - PDF_DENSITY_MIN),
        0.0,
        1.0,
    )
    pdf_radius = 1.08 + 0.34 * pdf_scaled
    ax.plot(
        pdf_radius * np.cos(theta),
        pdf_radius * np.sin(theta),
        color="#111827",
        linewidth=2.2,
    )

    # Mean direction at zero radians.
    ax.arrow(
        0,
        0,
        0.98,
        0,
        width=0.012,
        head_width=0.06,
        head_length=0.07,
        length_includes_head=True,
        color="#dc2626",
        zorder=5,
    )
    ax.scatter([1.0], [0.0], s=95, color="#dc2626", zorder=6)

    ax.set_title(f"Von Mises samples and pdf on the circle, kappa = {kappa:.2f}")
    ax.set_xlim(-1.55, 1.65)
    ax.set_ylim(-1.55, 1.55)
    ax.axis("off")

    colorbar = fig.colorbar(line, ax=ax, fraction=0.046, pad=0.02)
    colorbar.set_label("pdf density (fixed scale)")
    plt.show()


kappa_slider = FloatSlider(
    value=1.0,
    min=KAPPA_MIN,
    max=KAPPA_MAX,
    step=0.01,
    description="kappa",
    continuous_update=False,
    readout_format=".2f",
    style={"description_width": "initial"},
    layout={"width": "520px"},
)

out = interactive_output(draw_vonmises, {"kappa": kappa_slider})
display(VBox([kappa_slider, out]))